# Species Distribution Chart
Clips per species for the 4 families used in Level 1.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "utils" / "graph.py").is_file() and
           (PROJECT_ROOT / "v2" / "utils" / "splits.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Start Jupyter inside the cloned repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
metadata = pd.read_csv(os.path.join(PROJECT_ROOT, 'wmmd_metadata.csv'))

# Filter to the 4 families used in the project
L1_FAMILIES = ['Balaenidae', 'Delphinidae', 'Monodontidae', 'Phocidae']
subset = metadata[metadata['family'].isin(L1_FAMILIES)]

species_counts = subset.groupby(['family', 'species']).size().reset_index(name='clips')
species_counts = species_counts.sort_values(['family', 'clips'], ascending=[True, False])

print(f"Total clips: {species_counts['clips'].sum()}")
print(f"Species: {len(species_counts)}")
print(f"Families: {species_counts['family'].nunique()}")
print()
print(species_counts.to_string(index=False))
print()
print("Family totals:")
print(species_counts.groupby('family')['clips'].sum().to_string())

In [ ]:
FAMILY_COLORS = {
    'Balaenidae':   '#2166AC',
    'Delphinidae':  '#B2182B',
    'Monodontidae': '#F4A582',
    'Phocidae':     '#984EA3',
}

species_counts['display_name'] = species_counts['species'].str.replace('_', ' ')

fig, ax = plt.subplots(figsize=(9, 8))
colors = [FAMILY_COLORS[f] for f in species_counts['family']]
ax.barh(range(len(species_counts)), species_counts['clips'], color=colors, edgecolor='white', linewidth=0.5)

ax.set_yticks(range(len(species_counts)))
ax.set_yticklabels(species_counts['display_name'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Number of Clips', fontsize=11)
ax.set_title('Clips per Species (4 Families Used)', fontsize=13, fontweight='bold', color='#333333')

for i, val in enumerate(species_counts['clips']):
    ax.text(val + 1, i, str(val), va='center', fontsize=8, color='#555555')

legend_patches = [mpatches.Patch(color=c, label=f'{f} ({species_counts[species_counts["family"]==f]["clips"].sum()})')
                  for f, c in FAMILY_COLORS.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9, framealpha=0.9, title='Family (total)', title_fontsize=9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, species_counts['clips'].max() + 15)
plt.tight_layout()

save_dir = os.path.join(PROJECT_ROOT, 'figures')
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'species_distribution.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f"Saved to {save_path}")
plt.show()